# ArgLLM baseline (DF-QuAD, original prompts with facts) — Gemma 4 31B on StrategyClaim (500 claims)

Depth-one ArgLLM with the authors' prompts (argument mining and uncertainty evaluation, 0–100%, read after "Likelihood:", asked again up to three times when unreadable), with the annotators' facts added to every prompt. Strengths computed with the authors' DF-QuAD implementation (Uncertainpy). Results: 0.5 Base 86.6, Est. Base 93.4.

Run on Kaggle; the API key is read from Kaggle secrets (`OPENROUTER_API_KEY`).

In [1]:
!pip install -q openai datasets scikit-learn

import os, re, sys, time, pickle
from pathlib import Path
import numpy as np
import pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from datasets import load_from_disk

key = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
client = OpenAI(api_key=key,
                base_url="https://openrouter.ai/api/v1",
                timeout=180)
MODEL_NAME = "google/gemma-4-31b-it"
WORK_DIR = Path("/kaggle/working/gemma_dfquad_or")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Gemma models on OpenRouter:")
for m in client.models.list().data:
    if "gemma" in m.id.lower():
        print("  ", m.id)


def strip_thought(text):
    text = text or ""
    text = re.sub(r"<thought>.*?</thought>", "", text, flags=re.S)
    text = re.sub(r"^.*?</thought>", "", text, flags=re.S)
    return text.strip()


def call(prompt, temperature, max_tokens, top_p=None):
    kw = dict(model=MODEL_NAME, temperature=temperature,
              max_tokens=max_tokens + 1500,
              messages=[{"role": "user", "content": prompt}])
    if top_p is not None:
        kw["top_p"] = top_p
    last = None
    for a in range(20):
        try:
            r = client.chat.completions.create(**kw)
            t = strip_thought(r.choices[0].message.content)
            if t:
                return t
            raise ValueError("empty")
        except Exception as e:
            last = e
            if "402" in str(e):
                raise RuntimeError("no OpenRouter credit left")
            if "404" in str(e):
                raise RuntimeError("wrong MODEL_NAME, see list above")
            if "429" in str(e) or "rate" in str(e).lower():
                time.sleep(30)
            else:
                time.sleep(5)
    raise RuntimeError(f"failed: {last}")


print("test:", call("Is Paris in France? Answer true or false.",
                    0, 10))

Gemma models on OpenRouter:
   google/gemma-4-26b-a4b-it
   google/gemma-4-26b-a4b-it:free
   google/gemma-4-31b-it
   google/gemma-4-31b-it:free
   google/gemma-3-4b-it
   google/gemma-3-12b-it
   google/gemma-3-27b-it
   google/gemma-2-27b-it
test: True


In [2]:
repo = "/kaggle/working/argumentative-llms"
if not Path(repo).exists():
    os.system("cd /kaggle/working && git clone -q "
              "https://github.com/CLArg-group/argumentative-llms.git")
df = load_from_disk(repo + "/Datasets/StrategyQA/Experiment")
df = df.to_pandas().reset_index(drop=True)


def format_facts(f):
    if isinstance(f, (list, tuple, np.ndarray)):
        return "\n".join("- " + str(x) for x in f)
    return "" if f is None else str(f)


claims = df["claim"].astype(str).tolist()
facts = df["facts"].apply(format_facts).tolist()
gold = df["valid"].apply(lambda v: int(bool(v))).tolist()

sys.path.insert(0, repo)
import Uncertainpy.src.uncertainpy.gradual as grad
agg_f = grad.semantics.modular.ProductAggregation()
inf_f = grad.semantics.modular.LinearInfluence(conservativeness=1)

print("claims:", len(claims), "| true:", sum(gold))

claims: 500 | true: 250


In [3]:
def am_prompt(claim, fact, support):
    s = "supporting" if support else "attacking"
    v = "support" if support else "attack"
    return (f"Facts:\n{fact}\n\n"
            f"Please provide a single short argument {s} the following "
            "claim. Construct the argument so it refers to the truthfulness "
            "of the claim. Only provide an argument if you think there is a "
            f"valid and convincing {v} for this claim (there is a non-zero "
            "probability that the claim is true), otherwise return: N/A.\n"
            f"Claim: {claim}\n"
            "Now take a deep breath and come up with an argument.\n"
            "Argument:")


CALIB = ("definitely valid and values in between indicating various "
         "levels of uncertainty. Your estimates should be well-calibrated, "
         "so feel free to err on the side of caution and output moderate "
         "probabilities if you are not completely sure in your assessment. "
         "Please respond in the following form:\n\nLikelihood: The "
         "predicted likelihood that the considered ")


def ue_arg_prompt(claim, fact, arg, support):
    side = "in favour of" if support else "against"
    verb = "supports" if support else "refutes"
    return (f"Facts:\n{fact}\n\n"
            "You are an analyst evaluating the validity and relevance of "
            f'arguments. For the argument:\n\nArgument: "{arg}"\n\nplease '
            "give your confidence that the argument presents a compelling "
            f"case {side} the statement:"
            f'\n\nStatement: "{claim}"\n\nYour assessment should be based '
            f"on how well the argument {verb} the considered statement as "
            "well as the correctness, accuracy and truthfulness of the given "
            "argument. Your response should be between 0% and 100% with 0% "
            "indicating that the considered argument is definitely invalid, "
            "100% indicating that the considered argument is "
            + CALIB + "argument is valid")


def ue_topic_prompt(claim, fact):
    return (f"Facts:\n{fact}\n\n"
            "You are an analyst evaluating the validity of statements. For "
            f'the statement:\n\nStatement: "{claim}"\n\nplease give your '
            "confidence that the statement is correct, accurate and "
            "truthful. Your response should be between 0% and 100% with 0% "
            "indicating that the considered statement is definitely "
            "invalid, 100% indicating that the considered statement is "
            + CALIB + "statement is valid")


def gen_argument(claim, fact, support):
    t = call(am_prompt(claim, fact, support), 0.7, 150, top_p=0.95)
    if "N/A" in t or "n/a" in t:
        return "N/A"
    return t.replace("Argument:", "").strip().strip('"')


def read_lik(t):
    t = t or ""
    m = re.findall(r"Likelihood\W*(\d{1,3}(?:\.\d+)?)\s*%", t, re.I)
    if not m:
        m = re.findall(r"(\d{1,3}(?:\.\d+)?)\s*%", t)
    if m:
        return min(float(m[-1]), 100) / 100
    m = re.findall(r"Likelihood\W*(0?\.\d+|1\.0+|0|1)\b", t, re.I)
    if m:
        return float(m[-1])
    return None


def get_lik(prompt):
    for _ in range(3):
        v = read_lik(call(prompt, 0.7, 400, top_p=0.95))
        if v is not None:
            return v, 0
    return 0.5, 1


def strength(v, est):
    tau = v["topic_base"] if est else 0.5
    bag = grad.BAG()
    c = grad.Argument("db0", v["claim"], float(tau))
    bag.arguments["db0"] = c
    if v["sup"] != "N/A":
        bag.add_support(grad.Argument("S1", "s", float(v["sup_base"])), c)
    if v["att"] != "N/A":
        bag.add_attack(grad.Argument("A1", "a", float(v["att_base"])), c)
    grad.algorithms.computeStrengthValues(bag, agg_f, inf_f)
    return float(bag.arguments["db0"].strength)


def run_one_q(i):
    c, f = claims[i], facts[i]
    v = {"claim": c, "gold": gold[i], "unreadable": 0,
         "sup_base": None, "att_base": None}
    v["sup"] = gen_argument(c, f, True)
    v["att"] = gen_argument(c, f, False)
    if v["sup"] != "N/A":
        v["sup_base"], bad = get_lik(ue_arg_prompt(c, f, v["sup"], True))
        v["unreadable"] += bad
    if v["att"] != "N/A":
        v["att_base"], bad = get_lik(ue_arg_prompt(c, f, v["att"], False))
        v["unreadable"] += bad
    v["topic_base"], bad = get_lik(ue_topic_prompt(c, f))
    v["unreadable"] += bad
    v["s_05"] = strength(v, False)
    v["s_est"] = strength(v, True)
    return v


print("DF-QuAD ready")

DF-QuAD ready


In [4]:
import os, time, pickle
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

QK = WORK_DIR / "gemma_dfquad_500.pkl"
WORKERS = 4


def load_q():
    return pickle.load(open(QK, "rb")) if QK.exists() else {}


def save_q(d):
    tmp = QK.with_suffix(".tmp")
    pickle.dump(d, open(tmp, "wb"))
    tmp.replace(QK)


def safe_q(i):
    try:
        return i, run_one_q(i), None
    except Exception as e:
        return i, None, str(e)[:80]


def results_q(d):
    idx = sorted(d)
    g = [d[i]["gold"] for i in idx]
    rows = []
    for name, k in [("DF-QuAD 0.5 Base", "s_05"),
                    ("DF-QuAD Est. Base", "s_est")]:
        p = [int(d[i][k] > 0.5) for i in idx]
        tn, fp, fn, tp = confusion_matrix(g, p, labels=[0, 1]).ravel()
        rows.append({"method": name, "n": len(p),
                     "acc": round(accuracy_score(g, p) * 100, 1),
                     "f1": round(f1_score(g, p) * 100, 1),
                     "TN": tn, "FP": fp, "FN": fn, "TP": tp})
    print(pd.DataFrame(rows).to_string(index=False), flush=True)
    print("N/A sup:", sum(d[i]["sup"] == "N/A" for i in idx),
          "| N/A att:", sum(d[i]["att"] == "N/A" for i in idx),
          "| unreadable:", sum(d[i]["unreadable"] for i in idx),
          flush=True)


def run_pass_q(label):
    d = load_q()
    todo = [i for i in range(len(claims)) if i not in d]
    print(f"[{label}] have {len(d)}/500, running {len(todo)}",
          flush=True)
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futs = [pool.submit(safe_q, i) for i in todo]
        for f in as_completed(futs):
            i, v, err = f.result()
            if err:
                print(f"  error at {i}: {err}", flush=True)
                continue
            d[i] = v
            save_q(d)
            if len(d) % 50 == 0:
                mins = (time.time() - t0) / 60
                print(f"--- {len(d)}/500 | {mins:.0f} min ---")
                results_q(d)
    return load_q()


d = run_pass_q("main run")
for r in range(2):
    if len(d) < len(claims):
        time.sleep(30)
        d = run_pass_q(f"retry {r + 1}")

os.chdir("/kaggle/working")
rows = [dict(d[i], index=i) for i in sorted(d)]
pd.DataFrame(rows).to_csv("gemma_dfquad_500.csv", index=False)

print(f"\n=== FINAL: Gemma 4 31B - StrategyClaim - DF-QuAD "
      f"({len(d)}/500) ===")
results_q(d)

[main run] have 0/500, running 500
--- 50/500 | 3 min ---
           method  n  acc   f1  TN  FP  FN  TP
 DF-QuAD 0.5 Base 50 90.0 88.9  25   0   5  20
DF-QuAD Est. Base 50 92.0 91.3  25   0   4  21
N/A sup: 24 | N/A att: 7 | unreadable: 0
--- 100/500 | 6 min ---
           method   n  acc   f1  TN  FP  FN  TP
 DF-QuAD 0.5 Base 100 85.0 84.5  44   2  13  41
DF-QuAD Est. Base 100 91.0 91.1  45   1   8  46
N/A sup: 37 | N/A att: 14 | unreadable: 0
--- 150/500 | 8 min ---
           method   n  acc   f1  TN  FP  FN  TP
 DF-QuAD 0.5 Base 150 87.3 85.3  76   2  17  55
DF-QuAD Est. Base 150 92.7 91.9  77   1  10  62
N/A sup: 64 | N/A att: 23 | unreadable: 0
--- 200/500 | 12 min ---
           method   n  acc   f1  TN  FP  FN  TP
 DF-QuAD 0.5 Base 200 88.0 85.9 103   3  21  73
DF-QuAD Est. Base 200 93.5 92.7 104   2  11  83
N/A sup: 83 | N/A att: 31 | unreadable: 0
--- 250/500 | 15 min ---
           method   n  acc   f1  TN  FP  FN  TP
 DF-QuAD 0.5 Base 250 88.8 87.4 125   5  23  97
DF-QuAD 